# DPO Training on Qwen2.5-0.5B (Week 3)

Starts from **Week 2's SFT checkpoint** (not the raw base model) — DPO's theory assumes a reference policy `pi_ref` that already follows instructions reasonably well; DPO refines *preferences*, it isn't meant to teach instruction-following from scratch.

**Before running:** Settings → Accelerator → GPU T4 x2.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 1. Pull the repo and install dependencies

Includes the version-upgrade fix discovered while debugging `week3_peft_training.ipynb`: Kaggle's base image ships `transformers`/`torchao` versions too old for the current `trl`/`peft`, baked in here from the start this time instead of patched in after a failed run.

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
repo_url = f"https://{github_token}@github.com/zoom-BT/llm-alignment-internship.git"

!git clone --filter=blob:none --no-checkout {repo_url}
%cd llm-alignment-internship
!git sparse-checkout init --cone
!git sparse-checkout set src
!git checkout main
!pip install -q -r requirements.txt
!pip install -q -U transformers accelerate peft trl huggingface_hub torchao

## 2. Confirm the GPU is visible

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 3. Dry run (25 steps)

**Requires the Week 2 SFT checkpoint attached as input data** (Add Input → the `w02-sfttraining-qwen2-5-0-5b` notebook's output — same one used for the qualitative-analysis and after-eval notebooks), since we start DPO from that checkpoint, not the raw base model.

25 steps, not 20: `config.yaml`'s `dpo.eval_steps` is 20, so at least 25 guarantees one real evaluation fires before we trust the wiring — same reasoning as Week 2's SFT dry run.

In [ ]:
import yaml
from src.train import run_dpo

config = yaml.safe_load(open("config.yaml"))
sft_checkpoint = "/kaggle/input/notebooks/balbinotchoutzine/w02-sfttraining-qwen2-5-0-5b/llm-alignment-internship/results/checkpoints/final"

trainer = run_dpo(config, model_path=sft_checkpoint, max_steps=25)
print("Dry run finished without OOM, and eval ran at least once.")

## 4. Full DPO run

1000 training pairs, 1 epoch (`config.yaml`'s `dpo:` section) — small on purpose, per the contract's "small DPO experiment" wording. `config`/`sft_checkpoint` are still in memory from the cell above.

In [ ]:
trainer = run_dpo(config, model_path=sft_checkpoint)
print("DPO training complete. Model saved to results/dpo_checkpoints/final")

## 4a. Display the training curves

In [ ]:
from IPython.display import Image, display

display(Image(filename="results/dpo/training_curve.png"))

## 4b. Qualitative check — SFT vs. DPO, not perplexity

Deliberately **not** reusing `run_benchmark()` (test-set perplexity) here: DPO optimizes for *preference*, not next-token likelihood on held-out text — a model can become more preferred while its raw perplexity stays flat or even rises slightly (the same alignment-tax idea from InstructGPT). A few side-by-side generations are a more honest check of what DPO actually changed.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.evaluate import generate_samples
from src.utils import get_device

prompts_raw = [
    "Write a short story where a bear goes to the beach.",
    "What are the pros and cons of remote work?",
    "Explain how a car engine works.",
]

sft_tok = AutoTokenizer.from_pretrained(sft_checkpoint)
sft_model = AutoModelForCausalLM.from_pretrained(sft_checkpoint, dtype=torch.bfloat16)
sft_model.to(get_device())

prompts = [
    sft_tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
    for p in prompts_raw
]
with torch.no_grad():
    sft_completions = generate_samples(sft_model, sft_tok, prompts, max_new_tokens=150, do_sample=False)
del sft_model
torch.cuda.empty_cache()

with torch.no_grad():
    dpo_completions = generate_samples(trainer.model, trainer.processing_class, prompts, max_new_tokens=150, do_sample=False)

for p, before, after in zip(prompts_raw, sft_completions, dpo_completions):
    print("=" * 80)
    print(f"PROMPT: {p}")
    print(f"-- SFT (before DPO) --\n{before}")
    print(f"-- DPO (after) --\n{after}")

## 5. Next step

`results/dpo_checkpoints/final` now holds the DPO-tuned model, `results/dpo/training_curve.png` its training/eval loss curves. Together with `week3_peft_training.ipynb`'s LoRA result, this covers 2 of Week 3's 3 required practical methods (SFT+LoRA, DPO) — ORPO next, reusing this same reference-model pattern but without needing `ref_model` at all (its loss combines the SFT and preference terms directly).